<a href="https://colab.research.google.com/github/fboldt/aulas-am-bsi/blob/main/aula09a_sele%C3%A7%C3%A3o_de_caracter%C3%ADsticas.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [132]:
from sklearn.datasets import fetch_openml
from sklearn.model_selection import train_test_split
from sklearn.impute import SimpleImputer

arrhythmia = fetch_openml('arrhythmia', version=1)
X, y = arrhythmia.data, arrhythmia.target
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
imputer = SimpleImputer(strategy='mean')
X_train = imputer.fit_transform(X_train)
X_test = imputer.transform(X_test)
print(X_train.shape)

(361, 279)


In [133]:
from sklearn.linear_model import RidgeClassifier
model = RidgeClassifier()

def evaluate_model(model, data):
    X_train, X_test, y_train, y_test = data
    model.fit(X_train, y_train)
    return model.score(X_test, y_test)

data = (X_train, X_test, y_train, y_test)
print(evaluate_model(model, data))

0.5824175824175825


In [134]:
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import make_pipeline

model_std = make_pipeline(StandardScaler(), RidgeClassifier())

print(evaluate_model(model_std, data))

0.5164835164835165


## Redução dimensionalidade por extração de características a partir do domínio das características.

In [135]:
from sklearn.decomposition import PCA

pca = PCA(n_components=0.99)
model_pca = make_pipeline(pca, RidgeClassifier())
print(evaluate_model(model_pca, data))

0.7032967032967034


In [136]:
pca.fit(X_train)
n_feats = pca.n_components_
print(n_feats)

63


In [137]:
model_std_pca = make_pipeline(StandardScaler(), pca, RidgeClassifier())
print(evaluate_model(model_std_pca, data))

0.6593406593406593


In [138]:
model_pca_std = make_pipeline(pca, StandardScaler(), RidgeClassifier())
print(evaluate_model(model_pca_std, data))

0.7032967032967034


## Redução de dimensionalidade por seleção de características

* Pode melhorar o desempenho de classificação em situações que existam caracteríticas redundantes ou irrelevantes.
* Se não feita com cuidado resulta em overfiting e resultados enviesados.

## Ranking

### Filter

In [139]:
import numpy as np
from sklearn.base import BaseEstimator, TransformerMixin

def variance_score(X, y=None):
  return X.var(axis=0)[0]

def feature_ranking(X, y=None, score_function=variance_score):
  scores = []
  for f in range(X.shape[1]):
    score = score_function(X[:, f:f+1], y)
    scores.append(score)
  return np.argsort(scores)[::-1]

class FeatureRanking(BaseEstimator, TransformerMixin):
  def __init__(self, n_feats=None, score_function=variance_score):
    self.n_feats = n_feats
    self.score_function = score_function
  def fit(self, X, y=None):
    self.feats = feature_ranking(X, y, self.score_function)
    if self.n_feats is not None:
      self.feats = self.feats[:self.n_feats]
    return self
  def transform(self, X):
    return X[:,self.feats]

feat_rank = FeatureRanking(n_feats=10)
feat_rank.fit(X_train, y_train)
print(feat_rank.feats)

[ 10  13   9   5   2   7 248  12 258   6]


In [140]:
feat_rank = FeatureRanking(n_feats=n_feats)
model_feat_rank = make_pipeline(feat_rank, RidgeClassifier())
print(evaluate_model(model_feat_rank, data))

0.6703296703296703


In [141]:
model_std_feat_rank = make_pipeline(StandardScaler(), feat_rank, RidgeClassifier())
print(evaluate_model(model_std_feat_rank, data))

0.5824175824175825


In [142]:
model_feat_rank_std = make_pipeline(feat_rank, StandardScaler(), RidgeClassifier())
print(evaluate_model(model_feat_rank_std, data))

0.6703296703296703


In [143]:
model_feat_rank = make_pipeline(FeatureRanking(n_feats=80), RidgeClassifier())
print(evaluate_model(model_feat_rank, data))

0.6813186813186813


In [162]:
class FeatureHybridRanking(BaseEstimator, TransformerMixin):
  def __init__(self, model, score_function=variance_score):
    self.model = model
    self.score_function = score_function
  def fit(self, X, y=None):
    X_train, X_val, y_train, y_val = train_test_split(X, y, test_size=0.2, random_state=42)
    self.feats = feature_ranking(X, y, self.score_function)
    best_score = -np.inf
    best_n = 0
    for i in range(1,len(self.feats)+1):
      data = (X_train[:,:i], X_val[:,:i], y_train, y_val)
      score = evaluate_model(model, data)
      if score > best_score:
        best_score = score
        best_n = i
    self.feats = self.feats[:best_n]
    return self
  def transform(self, X):
    return X[:,self.feats]

model_hybr_rank = make_pipeline(FeatureHybridRanking(RidgeClassifier()), RidgeClassifier())
print(evaluate_model(model_hybr_rank, data))

0.6153846153846154


In [163]:
len(model_hybr_rank.steps[0][1].feats)

150

In [146]:
import numpy as np

TMP = np.array([
    [1,2,3],
    [2,5,0],
    [4,8,2],
    [2,3,6],
    [5,1,3]
])

def pairwise_distances(X1, X2=None):
  if X2 is None:
    X2 = X1
  return np.linalg.norm(X1[:, None, :] - X2[None, :, :], axis=-1)

print(pairwise_distances(TMP))

[[0.         4.35889894 6.78232998 3.31662479 4.12310563]
 [4.35889894 0.         4.12310563 6.32455532 5.83095189]
 [6.78232998 4.12310563 0.         6.70820393 7.14142843]
 [3.31662479 6.32455532 6.70820393 0.         4.69041576]
 [4.12310563 5.83095189 7.14142843 4.69041576 0.        ]]


In [147]:
distances = pairwise_distances(TMP)
distances[:,[1,2,4]][[1,2,4],:]

array([[0.        , 4.12310563, 5.83095189],
       [4.12310563, 0.        , 7.14142843],
       [5.83095189, 7.14142843, 0.        ]])

In [148]:
distances[:,[1,2,4]][[1,2,4],:].mean()

np.float64(3.7989968775568475)

In [149]:
distances[:,[0,3]][[0,3],:].mean()

np.float64(1.6583123951777)

In [150]:
for i in range(TMP.shape[1]):
  print(f"feature {i}")
  dists = pairwise_distances(TMP[:,i:i+1])
  print(dists)
  print(f"mean distances: {dists.mean()}")
  print()

feature 0
[[0. 1. 3. 1. 4.]
 [1. 0. 2. 0. 3.]
 [3. 2. 0. 2. 1.]
 [1. 0. 2. 0. 3.]
 [4. 3. 1. 3. 0.]]
mean distances: 1.6

feature 1
[[0. 3. 6. 1. 1.]
 [3. 0. 3. 2. 4.]
 [6. 3. 0. 5. 7.]
 [1. 2. 5. 0. 2.]
 [1. 4. 7. 2. 0.]]
mean distances: 2.72

feature 2
[[0. 3. 1. 3. 0.]
 [3. 0. 2. 6. 3.]
 [1. 2. 0. 4. 1.]
 [3. 6. 4. 0. 3.]
 [0. 3. 1. 3. 0.]]
mean distances: 2.08



In [151]:
def intra_class_mean_distance(X, y):
  mean_dist = 0
  for label in np.unique(y):
    X_label = X[y==label]
    dists = pairwise_distances(X_label)
    mean_dist += dists.mean()
  return mean_dist / len(np.unique(y))

print(intra_class_mean_distance(TMP, np.array([0,1,1,0,1])))

2.7286546363672737


In [152]:
def inter_class_mean_distance(X, y):
  mean_dist = 0
  for label in np.unique(y):
    in_class = X[y == label]
    out_class = X[y != label]
    dists = pairwise_distances(in_class, out_class)
    mean_dist += dists.mean()
  return mean_dist / len(np.unique(y))

print(inter_class_mean_distance(TMP, np.array([0,1,1,0,1])))


5.49791826082386


In [164]:
def distance_score(X, y):
  return inter_class_mean_distance(X, y) / (intra_class_mean_distance(X, y)+1e-8)

print(distance_score(TMP, np.array([0,1,1,0,1])))

2.0148824139923227


In [165]:
feat_rank_dist = FeatureHybridRanking(RidgeClassifier(), distance_score)
model_frd = make_pipeline(feat_rank_dist, RidgeClassifier())
print(evaluate_model(model_frd, data))

0.6703296703296703


In [166]:
len(model_frd.steps[0][1].feats)

150

In [167]:
def wrapper_ridge_score(X, y):
  X_tr, X_val, y_tr, y_val = train_test_split(X, y, test_size=0.2, random_state=42)
  return RidgeClassifier().fit(X_tr, y_tr).score(X_val, y_val)

feat_rank_wrapper = FeatureHybridRanking(RidgeClassifier(), wrapper_ridge_score)
model_fw = make_pipeline(feat_rank_wrapper, RidgeClassifier())
print(evaluate_model(model_fw, data))

0.6373626373626373


In [168]:
len(model_fw.steps[0][1].feats)

150

In [193]:
class SequentialForwardSelection(BaseEstimator, TransformerMixin):
  def __init__(self, n_feats=None, score_function=variance_score):
    self.n_feats = n_feats
    self.score_function = score_function
  def fit(self, X, y=None):
    self.feats = []
    if self.n_feats is None:
      self.n_feats = X.shape[1]
    remaining_feats = list(range(X.shape[1]))
    while len(self.feats) < self.n_feats:
      best_score = -np.inf
      best_f = None
      for f in remaining_feats:
        self.feats.append(f)
        score = self.score_function(X[:,self.feats], y)
        self.feats.remove(f)
        if score > best_score:
          best_score = score
          best_f = f
      self.feats.append(best_f)
      remaining_feats.remove(best_f)
    return self
  def transform(self, X, y=None):
    return X[:,self.feats]

feat_sfs = SequentialForwardSelection(n_feats=10)
feat_sfs.fit(X_train, y_train)
print(feat_sfs.feats)

[10, 0, 1, 2, 3, 4, 5, 6, 7, 8]


In [177]:
feat_sfs = SequentialForwardSelection(n_feats=n_feats)
model_sfs = make_pipeline(feat_sfs, RidgeClassifier())
print(evaluate_model(model_sfs, data))

0.5164835164835165


In [184]:
print((model_sfs.steps[0][1].feats))
print(len(model_sfs.steps[0][1].feats))

[10, 0, 1, 2, 3, 4, 5, 6, 7, 8, 9, 11, 12, 13, 14, 15, 16, 17, 18, 19, 20, 21, 22, 23, 24, 25, 26, 27, 28, 29, 30, 31, 32, 33, 34, 35, 36, 37, 38, 39, 40, 41, 42, 43, 44, 45, 46, 47, 48, 49, 50, 51, 52, 53, 54, 55, 56, 57, 58, 59, 60, 61, 62]
63


In [186]:
feat_sfsds = SequentialForwardSelection(n_feats=5, score_function=distance_score)
model_sfsds = make_pipeline(feat_sfsds, RidgeClassifier())
print(evaluate_model(model_sfsds, data))

0.4835164835164835


In [194]:
class HybridSFS(BaseEstimator, TransformerMixin):
  def __init__(self, model, score_function=variance_score):
    self.model = model
    self.score_function = score_function
  def fit(self, X, y=None):
    X_train, X_val, y_train, y_val = train_test_split(X, y, test_size=0.2, random_state=42)
    SFS = SequentialForwardSelection(score_function=self.score_function)
    SFS.fit(X_train, y_train)
    self.feats = SFS.feats
    best_score = -np.inf
    best_n = 0
    for i in range(1,len(self.feats)+1):
      data = (X_train[:,:i], X_val[:,:i], y_train, y_val)
      score = evaluate_model(model, data)
      if score > best_score:
        best_score = score
        best_n = i
    self.feats = self.feats[:best_n]
    return self
  def transform(self, X):
    return X[:,self.feats]

feat_hsfs = HybridSFS(RidgeClassifier())
model_hsfs = make_pipeline(feat_hsfs, RidgeClassifier())
print(evaluate_model(model_hsfs, data))

0.6043956043956044


In [195]:
print((model_hsfs.steps[0][1].feats))
print(len(model_hsfs.steps[0][1].feats))

[10, 0, 1, 2, 3, 4, 5, 6, 7, 8, 9, 11, 12, 13, 14, 15, 16, 17, 18, 19, 20, 21, 22, 23, 24, 25, 26, 27, 28, 29, 30, 31, 32, 33, 34, 35, 36, 37, 38, 39, 40, 41, 42, 43, 44, 45, 46, 47, 48, 49, 50, 51, 52, 53, 54, 55, 56, 57, 58, 59, 60, 61, 62, 63, 64, 65, 66, 67, 68, 69, 70, 71, 72, 73, 74, 75, 76, 77, 78, 79, 80, 81, 82, 83, 84, 85, 86, 87, 88, 89, 90, 91, 92, 93, 94, 95, 96, 97, 98, 99, 100, 101, 102, 103, 104, 105, 106, 107, 108, 109, 110, 111, 112, 113, 114, 115, 116, 117, 118, 119, 120, 121, 122, 123, 124, 125, 126, 127, 128, 129, 130, 131, 132, 133, 134, 135, 136, 137, 138, 139, 140, 141, 142, 143, 144, 145, 146, 147, 148, 149]
150
